# RegimeShift: Macro-Aware Tactical Asset Allocation Engine
### Summer of Quant — Advanced Capstone Submission

This notebook runs the **full pipeline top to bottom**:

`data → features → regime detection (HMM) → convex optimization (cvxpy) → walk-forward backtest → results`

It is a thin, narrated wrapper around the actual implementation in `src/`
(so the logic is unit-testable and reusable, rather than living only inside
notebook cells). Every design decision referenced below is explained in more
depth in `README.md`.

**Contents**
1. Setup
2. Data — multi-asset universe (NSE equity, gilt bonds, gold, India VIX)
3. Feature engineering (momentum, volatility, VIX signals)
4. Walk-forward split design
5. Regime detection: fitting & interpreting the HMM
6. Regime → portfolio optimization (cvxpy)
7. The lookahead-bias trap, and how this pipeline avoids it
8. Full walk-forward backtest
9. Results: regime overlay, transition matrix, equity curves, drawdowns, performance table
10. Robustness checks (causal vs batch decoding, rolling vs expanding windows)


## 1. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # so `src` is importable when run from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data_loader
from src import features as feat_mod
from src import regime_hmm
from src import walkforward
from src import backtest as bt_mod
from src import optimizer as opt_mod
from src import metrics

plt.rcParams["figure.figsize"] = (12, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 120)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set to True only to smoke-test the pipeline without a network connection.
# The results discussed in the README were produced with USE_SYNTHETIC = False.
USE_SYNTHETIC = False

print("Setup complete.")


## 2. Data

**Universe** (three genuinely different asset classes, plus a volatility feature):

| Role | Ticker | What it is |
|---|---|---|
| Equity | `^NSEI` | Nifty 50 index |
| Bonds | `NETFLTGILT.NS` (fallback `LIQUIDBEES.NS`) | Long-duration government-bond ETF — actually rate-sensitive, unlike a cash-like liquid fund |
| Gold | `GOLDBEES.NS` | Most liquid NSE gold ETF |
| Vol feature | `^INDIAVIX` | India VIX — feature only, **not** in the optimizable portfolio |

`data_loader.load_universe()` inner-joins all series onto a common trading-day
index and prints a NaN check post-alignment, since different exchanges/assets
have different holiday calendars and gaps.


In [ ]:
if USE_SYNTHETIC:
    prices, vix = data_loader.generate_synthetic_universe(start="2010-01-01", end="2025-01-01")
else:
    prices, vix = data_loader.load_universe(start="2010-01-01", end="2025-01-01")

prices.tail()


In [ ]:
fig, ax = plt.subplots()
(prices / prices.iloc[0]).plot(ax=ax)
ax.set_title("Rebased Asset Prices (=1 at start)")
ax.set_ylabel("Growth of 1 unit")
plt.show()

fig, ax = plt.subplots()
vix.plot(ax=ax, color="firebrick")
ax.set_title("India VIX")
plt.show()


## 3. Feature Engineering

Three feature families (see `src/features.py` docstring for the full
rationale): **momentum** (direction), **volatility** (uncertainty — the more
reliable crisis signal), and **VIX level/change** (a forward-looking,
market-implied fear gauge, independent of realized-return-based features).
Every window used below is a `.rolling(N)` or `.diff(N)` — strictly
backward-looking by construction, which is the first line of defense against
lookahead bias (Section 7 makes this rigorous).


In [ ]:
feat_raw = feat_mod.build_feature_frame(prices, vix)
print(f"{feat_raw.shape[0]} rows x {feat_raw.shape[1]} feature columns")
print(f"Range: {feat_raw.index[0].date()} -> {feat_raw.index[-1].date()}")
feat_raw[feat_mod.HMM_FEATURE_COLS].tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(feat_raw.index, feat_raw["vol_21d"], label="21d realized vol (ann.)")
axes[0].plot(feat_raw.index, feat_raw["vix_level"] / 100, label="India VIX / 100 (rescaled)")
axes[0].legend(); axes[0].set_title("Volatility features — sanity check against known stress periods")

axes[1].plot(feat_raw.index, feat_raw["mom_21d"], label="21d momentum", alpha=0.8)
axes[1].plot(feat_raw.index, feat_raw["mom_126d"], label="126d momentum", alpha=0.8)
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].legend(); axes[1].set_title("Momentum features at two horizons")
plt.tight_layout()
plt.show()


## 4. Walk-Forward Split Design

We use **expanding-window** walk-forward splits (`src/walkforward.py`):
the training window always starts at observation 0 and grows; the test
window is a fixed ~6-month (126 trading day) block immediately after it.
Rationale: our history isn't long enough to comfortably throw away data in a
rolling window, and market regimes recur — a well-estimated Crisis emission
distribution from one crash is still useful context for detecting the next
one.

The very first `min_train_size` observations are **never scored** — the HMM
needs that much history just to be fit for the first time, and scoring
predictions made "before the model existed" would itself be a form of
lookahead bias.


In [ ]:
splits = walkforward.expanding_walk_forward_splits(
    n_obs=len(feat_raw), n_splits=6, min_train_size=750, test_size=126,
)

fig, ax = plt.subplots(figsize=(12, 2.6))
for i, (train_idx, test_idx) in enumerate(splits):
    ax.barh(i, train_idx[-1] - train_idx[0], left=train_idx[0], color="steelblue",
            label="train" if i == 0 else "")
    ax.barh(i, test_idx[-1] - test_idx[0], left=test_idx[0], color="firebrick",
            label="test" if i == 0 else "")
ax.set_yticks(range(len(splits))); ax.set_yticklabels([f"Fold {i+1}" for i in range(len(splits))])
ax.set_xlabel("Observation index (time)")
ax.set_title(f"{len(splits)} Expanding-Window Walk-Forward Folds")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

for i, (tr, te) in enumerate(splits):
    print(f"Fold {i+1}: train {feat_raw.index[tr[0]].date()}..{feat_raw.index[tr[-1]].date()}"
          f"  |  test {feat_raw.index[te[0]].date()}..{feat_raw.index[te[-1]].date()}")


## 5. Regime Detection: Fitting & Interpreting the HMM

Quick, single-fold demo of the mechanics (fit → predict → label) before
running the full walk-forward loop, so each piece is inspectable on its own.
This cell fits on the **first fold's training data only** — it is *not* the
final regime history (that comes from the full walk-forward run in Section
8, which re-fits inside every fold).

**Why `n_components=3`, why `covariance_type="diag"`:** see the docstring in
`src/regime_hmm.py` — short version: 3 is the smallest model that actually
distinguishes Bull/Bear/Crisis as the brief asks; `diag` keeps Baum-Welch
stable on the smaller early training folds (only ~6 features, as few as
~750 training observations in fold 1).


In [ ]:
train_idx0, test_idx0 = splits[0]
train_feat0 = feat_raw.iloc[train_idx0][feat_mod.HMM_FEATURE_COLS]

mu0, sigma0 = feat_mod.fit_scaler(train_feat0)
X0 = feat_mod.apply_scaler(train_feat0, mu0, sigma0).values

demo_model = regime_hmm.fit_hmm(X0, n_states=3)
demo_states = demo_model.predict(X0)
demo_label_map = regime_hmm.label_states(demo_model, feat_raw.iloc[train_idx0], demo_states)

print("State -> regime label mapping (fold 1, train only):", demo_label_map)
print("\nTransition matrix (fold 1, train only):")
regime_hmm.transition_matrix_df(demo_model, demo_label_map)


In [ ]:
demo_df = feat_raw.iloc[train_idx0].copy()
demo_df["state"] = demo_states
demo_df["regime"] = demo_df["state"].map(demo_label_map)

fig, ax = plt.subplots(figsize=(13, 5))
px0 = prices.loc[demo_df.index, "EQUITY"]
ax.plot(px0.index, px0.values, color="black", lw=1.0, zorder=3)
for label, color in regime_hmm.REGIME_COLORS.items():
    mask = demo_df["regime"] == label
    ax.scatter(px0.index[mask], px0.values[mask], s=6, color=color, label=label, zorder=2)
ax.legend(loc="upper left")
ax.set_title("Single-fold demo fit (fold 1 training data only) — NOT the final out-of-sample result")
plt.show()


## 6. Regime → Portfolio Optimization (cvxpy)

Each regime maps to a different convex objective, solved long-only /
fully-invested with `cvxpy` (see `src/optimizer.py` docstring for the full
"why"):

| Regime | Objective | Constraints |
|---|---|---|
| **Bull** | Maximize Sharpe ratio (tangency portfolio, convex QP reformulation) | long-only, sum=1 |
| **Bear** | Minimize variance | long-only, sum=1, EQUITY ≤ 45% |
| **Crisis** | Minimize variance | long-only, sum=1, EQUITY ≤ 15%, GOLD ≥ 30% |

A quick sanity check with made-up but reasonable moments:


In [ ]:
mu_demo = np.array([0.10, 0.04, 0.03])       # illustrative annualized EQUITY/BONDS/GOLD returns
Sigma_demo = np.array([
    [0.030, -0.003,  0.001],
    [-0.003, 0.006,  0.001],
    [0.001,  0.001,  0.015],
])

for regime in ["Bull", "Bear", "Crisis"]:
    w = opt_mod.solve_regime_weights(regime, mu_demo, Sigma_demo, opt_mod.ASSET_ORDER)
    print(f"{regime:8s} -> " + ", ".join(f"{a}={x:.1%}" for a, x in zip(opt_mod.ASSET_ORDER, w)))


## 7. The Lookahead-Bias Trap (and how this pipeline avoids it)

Four places it can sneak in, and how each is closed off here:

| Source | The leak | Fix used in this repo |
|---|---|---|
| Feature scaling | Z-scoring with full-sample mean/std | `features.fit_scaler` is fit on **train only** inside each fold, then applied to that fold's test slice (`backtest.py`) |
| HMM fitting | Fitting once on the full dataset, "predicting" the past | HMM is **re-fit inside every walk-forward fold**, train data only |
| Regime decoding at test time | `model.predict(test_fold)` runs Viterbi *smoothing* over the whole test block at once — day 5's label could be influenced by day 120 | `regime_hmm.causal_predict_regimes` decodes the **growing prefix** (train + test-so-far) at every rebalance date and keeps only the last state — day *t*'s label is a function of data ≤ *t* only |
| Moment estimation (mu, Sigma) for the optimizer | Using a window that includes the rebalance date itself or later | `backtest._estimate_moments` explicitly excludes the "as-of" date and only looks backward |

The cell below reproduces the core demonstration from the project guide: a
z-score computed with full-sample statistics is measurably distorted **even
in the pre-crash period**, purely because a crash exists later in the
sample that the "leaky" calculation already has access to.


In [ ]:
np.random.seed(7)
n = 300
calm = np.random.normal(0.0005, 0.006, n - 30)
crash = np.random.normal(-0.01, 0.04, 30)
toy_returns = pd.Series(np.concatenate([calm, crash]))
toy_vol = toy_returns.rolling(21).std() * np.sqrt(252)

leaky_z = (toy_vol - toy_vol.mean()) / toy_vol.std()
safe_mean = toy_vol.expanding(min_periods=40).mean()
safe_std = toy_vol.expanding(min_periods=40).std()
safe_z = (toy_vol - safe_mean) / safe_std

fig, ax = plt.subplots()
ax.plot(leaky_z, label="Leaky z-score (full-sample stats)")
ax.plot(safe_z, label="Safe z-score (expanding / train-only stats)")
ax.axvline(n - 30, color="red", ls="--", label="crash begins here")
ax.legend()
ax.set_title("The leaky z-score is distorted even BEFORE the crash happens")
plt.show()

print("Leaky pre-crash mean:", round(leaky_z.iloc[:100].mean(), 4))
print("Safe  pre-crash mean:", round(safe_z.iloc[:100].mean(), 4))


## 8. Full Walk-Forward Backtest

This is the real run: every fold refits the HMM on train-only data, labels
its states from train-only statistics, and — for every scheduled rebalance
date in that fold's test window — causally decodes the regime, estimates
`mu`/`Sigma` from a strictly-historical trailing window, and solves the
regime-appropriate `cvxpy` optimization. Portfolio weights drift with daily
returns between rebalances; turnover is tracked per rebalance so transaction
costs can be applied after the fact at any bps level.

Rebalance cadence: every 5 trading days (~weekly) — frequent enough to react
to a genuine regime flip within about a week, infrequent enough that
turnover/costs stay realistic.


In [ ]:
result = bt_mod.run_walk_forward_backtest(
    prices, feat_raw, splits,
    rebalance_freq=bt_mod.DEFAULT_REBALANCE_FREQ,   # 5 trading days
    mu_sigma_lookback=bt_mod.DEFAULT_MU_SIGMA_LOOKBACK,  # 126 trading days
)

TC_BPS = bt_mod.DEFAULT_TC_BPS  # 7.5 bps per unit of turnover — within the 5-10bps spec range
net_returns = bt_mod.apply_transaction_costs(result["gross_returns"], result["turnover"], tc_bps=TC_BPS)

test_index = result["test_index"]
bench_6040 = bt_mod.static_benchmark_returns(prices, test_index, "60_40")
bench_eq = bt_mod.static_benchmark_returns(prices, test_index, "equal_weight")

print(f"Out-of-sample backtest window: {test_index[0].date()} -> {test_index[-1].date()}  "
      f"({len(test_index)} trading days)")


## 9. Results

### 9.1 Regime overlay (out-of-sample, causal)

In [ ]:
regime_daily = result["regime_daily"].dropna()
px = prices.loc[regime_daily.index, "EQUITY"]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(px.index, px.values, color="black", lw=1.0, zorder=3)
for label, color in regime_hmm.REGIME_COLORS.items():
    mask = regime_daily == label
    ax.scatter(px.index[mask], px.values[mask], s=8, color=color, label=label, zorder=2)
ax.legend(loc="upper left")
ax.set_title("Out-of-Sample HMM Regimes Overlaid on EQUITY Price (walk-forward, causal decoding)")
plt.show()

regime_daily.value_counts(normalize=True).rename("share of out-of-sample days")


### 9.2 Transition matrix (final fold's fitted model — the most data-informed)

In [ ]:
final_fold = result["fold_models"][-1]
tm = final_fold["transition_matrix"]
print(f"From fold {final_fold['fold']+1}, label map = {final_fold['label_map']}")
tm.style.background_gradient(cmap="Blues", vmin=0, vmax=1).format("{:.3f}")


### 9.3 Equity curves — strategy vs. static benchmarks

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for s, label, style in [
    (result["gross_returns"], "Regime-Adaptive (gross, no costs)", "--"),
    (net_returns, "Regime-Adaptive (net of costs)", "-"),
    (bench_6040, "Static 60/40", "-"),
    (bench_eq, "Equal-Weight", "-"),
]:
    ax.plot(metrics.equity_curve(s).index, metrics.equity_curve(s).values, style, label=label, lw=1.6)
ax.set_title("Out-of-Sample Equity Curves")
ax.set_ylabel("Growth of ₹1")
ax.legend()
plt.show()


### 9.4 Drawdowns

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
for s, label in [
    (net_returns, "Regime-Adaptive (net)"),
    (bench_6040, "Static 60/40"),
    (bench_eq, "Equal-Weight"),
]:
    dd = metrics.drawdown_series(s)
    ax.fill_between(dd.index, dd.values, 0, alpha=0.3, label=label)
    ax.plot(dd.index, dd.values, lw=0.8)
ax.legend()
ax.set_title("Drawdown (Underwater Plot)")
plt.show()


### 9.5 Portfolio weights over time

In [ ]:
w = result["weights"]
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.stackplot(w.index, w["EQUITY"], w["BONDS"], w["GOLD"], labels=["EQUITY", "BONDS", "GOLD"], alpha=0.85)
ax.set_ylim(0, 1)
ax.legend(loc="upper left")
ax.set_title("Regime-Adaptive Portfolio Weights Over Time")
plt.show()


### 9.6 Performance summary

In [ ]:
summary_rows = [
    metrics.summarize(result["gross_returns"], result["weights"], "Regime-Adaptive (gross)"),
    metrics.summarize(net_returns, result["weights"], "Regime-Adaptive (net of costs)"),
    metrics.summarize(bench_6040, None, "Static 60/40"),
    metrics.summarize(bench_eq, None, "Equal-Weight"),
]
summary_df = pd.DataFrame(summary_rows).set_index("Strategy")
summary_df.round(4)


In [ ]:
os.makedirs("../outputs", exist_ok=True)
summary_df.to_csv("../outputs/performance_summary.csv")
result["regime_daily"].to_csv("../outputs/regime_labels.csv")
tm.to_csv("../outputs/transition_matrix.csv")
print("Saved performance_summary.csv, regime_labels.csv, transition_matrix.csv to ../outputs/")


## 10. Robustness Checks

### 10.1 Causal vs. batch Viterbi decoding — does it actually matter?

`regime_hmm.py` provides both `causal_predict_regimes` (used everywhere
above — decodes the growing prefix, strictly no future leakage within a
fold) and `batch_predict_regimes` (decodes the whole test fold in one
Viterbi pass — faster, but a day's label can be influenced by later days in
the *same* fold). We compare them on the last fold to see how much the
smoothing actually changes labels in practice.


In [ ]:
train_idx_last, test_idx_last = splits[-1]
train_feat_last = feat_raw.iloc[train_idx_last][feat_mod.HMM_FEATURE_COLS]
test_feat_last = feat_raw.iloc[test_idx_last][feat_mod.HMM_FEATURE_COLS]

mu_l, sigma_l = feat_mod.fit_scaler(train_feat_last)
train_scaled_l = feat_mod.apply_scaler(train_feat_last, mu_l, sigma_l).values
test_scaled_l = feat_mod.apply_scaler(test_feat_last, mu_l, sigma_l).values

model_l = regime_hmm.fit_hmm(train_scaled_l, n_states=3)
train_states_l = model_l.predict(train_scaled_l)
label_map_l = regime_hmm.label_states(model_l, feat_raw.iloc[train_idx_last], train_states_l)

rebal_pos = np.arange(0, len(test_idx_last), bt_mod.DEFAULT_REBALANCE_FREQ)
causal_states = regime_hmm.causal_predict_regimes(model_l, train_scaled_l, test_scaled_l, rebal_pos)
batch_states_full = regime_hmm.batch_predict_regimes(model_l, test_scaled_l)
batch_states = batch_states_full[rebal_pos]

causal_labels = [label_map_l[s] for s in causal_states]
batch_labels = [label_map_l[s] for s in batch_states]

agreement = np.mean(np.array(causal_labels) == np.array(batch_labels))
print(f"Causal vs. batch decoding agreement on last fold: {agreement:.1%}")
print("Causal:", causal_labels)
print("Batch: ", batch_labels)


### 10.2 Expanding vs. rolling walk-forward windows

A quick structural comparison of fold boundaries under the alternative
(rolling) window scheme — see `src/walkforward.py`. We use expanding by
default for the reasons given in Section 4; this cell just visualizes how
differently the folds would be laid out under a fixed-size rolling window,
as a documented alternative rather than a full second backtest run.


In [ ]:
rolling_splits = walkforward.rolling_walk_forward_splits(
    n_obs=len(feat_raw), n_splits=6, train_size=750, test_size=126,
)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for i, (train_idx, test_idx) in enumerate(splits):
    axes[0].barh(i, train_idx[-1]-train_idx[0], left=train_idx[0], color="steelblue")
    axes[0].barh(i, test_idx[-1]-test_idx[0], left=test_idx[0], color="firebrick")
axes[0].set_title("Expanding window (used for the reported results)")

for i, (train_idx, test_idx) in enumerate(rolling_splits):
    axes[1].barh(i, train_idx[-1]-train_idx[0], left=train_idx[0], color="steelblue")
    axes[1].barh(i, test_idx[-1]-test_idx[0], left=test_idx[0], color="firebrick")
axes[1].set_title("Rolling window (alternative)")
axes[1].set_xlabel("Observation index (time)")
plt.tight_layout()
plt.show()


## Conclusion

This notebook — together with `src/` and `main.py` — satisfies every item on
the project checklist:

- ✅ HMM-based Bull/Bear/Crisis regime classifier fit only on NSE-derived
  data, with **no manually labelled days** (labels come from a
  volatility/momentum ranking heuristic applied to the model's *own* fitted
  state assignments).
- ✅ Portfolio weights that shift by detected regime, with a different
  convex objective per regime.
- ✅ Walk-forward validation where the HMM is refit on **past data only** at
  every fold, plus **causal (prefix-only) Viterbi decoding** at test time —
  stricter than the simple batch-decode-per-fold sketch in the project
  guide.
- ✅ Transaction costs (7.5 bps, inside the 5–10 bps spec) explicitly
  modelled, with results shown gross **and** net of costs.
- ✅ Comparison against static 60/40 and equal-weight benchmarks on Sharpe,
  Sortino, max drawdown, Calmar, and turnover.

See `README.md` for the full list of key decisions and instructions to
reproduce these results.
